# LLM Finetuning Cookbook with NeMo AutoModel

This notebook is a **finetuning recipe** that walks through:

1. **Environment setup** — Use the existing repo and environment at `~/Automodel`.
2. **Dataset creation** — Build training, validation, and test sets in the JSONL format expected by the finetuning script (`input` / `output` fields).
3. **Finetuning** — Run PEFT (LoRA) finetuning using `examples/llm_finetune/finetune.py` and `examples/llm_finetune/nemotron/base-peft-config-cookbook.yaml`.

**Requirements:** Multi-GPU machine (e.g. 8× H100 for [Nemotron-3-Super](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16). For smaller models, fewer GPUs suffice. Hugging Face authentication is required for gated models (e.g. Nemotron-3-Super).

---
## 1. Environment setup

### 1.1 Set up the Workspace ROOT and Data Paths

In [11]:
# Configuration — adjust these to your setup
import os
import sys

# Detect working environment and Adding workspace ROOT to the path
ON_BREV = os.path.exists("/ephemeral") or os.environ.get("BREV", "")

if ON_BREV:
    WORKSPACE_ROOT = "/ephemeral/shadeform"
    os.makedirs(WORKSPACE_ROOT, exist_ok=True)
    os.chdir(WORKSPACE_ROOT)
else:
    WORKSPACE_ROOT = os.getcwd()

# All steps run from AUTOMODEL_DIR.
AUTOMODEL_DIR = os.path.join(WORKSPACE_ROOT, "Automodel")  # existing  Automodel repo and .venv
DATASET_DIR = os.path.join(WORKSPACE_ROOT, "new_results")  # training.jsonl, validation.jsonl, test.jsonl

os.makedirs(WORKSPACE_ROOT, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"Workspace ROOT: {WORKSPACE_ROOT}")
print(f"AutoModel (working dir): {AUTOMODEL_DIR}")
print(f"Datasets: {DATASET_DIR}")

Workspace ROOT: /ephemeral/shadeform
AutoModel (working dir): /ephemeral/shadeform/Automodel
Datasets: /ephemeral/shadeform/new_results


### 1.2 Set up the Working Directory

In [14]:
assert os.path.isdir(AUTOMODEL_DIR), f"Expected existing repo at {AUTOMODEL_DIR}"
print(f"Automodel Directory exists: {AUTOMODEL_DIR}")
if os.path.realpath(os.getcwd()) != os.path.realpath(AUTOMODEL_DIR):
    os.chdir(AUTOMODEL_DIR)
print(f"Working from: {os.getcwd()}")
assert os.getcwd() == os.path.abspath(AUTOMODEL_DIR), "Working directory does not match AUTOMODEL_DIR"

Automodel Directory exists: /ephemeral/shadeform/Automodel
Working from: /ephemeral/shadeform/Automodel


### 1.3 Set up the Working Environment

In [15]:
venv_dir = os.path.join(AUTOMODEL_DIR, ".venv")

if not os.path.isdir(venv_dir):
    repo_exists = os.path.isdir(AUTOMODEL_DIR) and os.path.exists(os.path.join(AUTOMODEL_DIR, ".git"))
    if not repo_exists:
        print(f"AutoModel repo not found at {AUTOMODEL_DIR}. Check the README.md to clone the repo.")

assert os.path.isdir(AUTOMODEL_DIR), f"Expected existing repo at {AUTOMODEL_DIR}"
assert os.path.isdir(venv_dir), f"Expected existing .venv at {venv_dir}"
print(f"Using existing env: {venv_dir}")

Using existing env: /ephemeral/shadeform/Automodel/.venv


---
**Optional (Nemotron-Super / Mamba):** If you use the Nemotron-Super checkpoint and hit ABI issues with `mamba-ssm`, install it with `--no-build-isolation` so it is built against your venv's PyTorch:

```bash
cd "$AUTOMODEL_DIR" && source .venv/bin/activate
pip install mamba-ssm --no-build-isolation
```

**Hugging Face:** For gated models (e.g. `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16`), run `huggingface-cli login` in the same environment before finetuning.

---
## 2. Dataset creation

In this cookbook, the BIRD-SQL dataset is used for fine-tuning Nemotron-3-super with NeMo AutoModel.

#### A Little about the dataset:
[BIRD-SQL](https://bird-bench.github.io/) is a large-scale text-to-SQL benchmark that focuses on real-world conditions: it has 12,751 question–SQL pairs over 95 databases from 37 domains (about 33.4 GB total). Unlike benchmarks that only stress schema understanding, BIRD-SQL uses realistic, “dirty” data and requires reasoning about database contents and external knowledge, making it a good fit for training models for practical text-to-SQL.

The cells below load the BIRD-SQL, convert it to input/output JSONL, and write `training.jsonl`, `validation.jsonl`, and `test.jsonl` to `DATASET_DIR`.

The config in the next cell controls the split: `VALIDATION_FRACTION` and `TEST_FRACTION` set aside validation and test data, `SPLIT_SEED` fixes reproducibility, `INCLUDE_REASONING` toggles reasoning text (vs. `BIRD-only`, like `--no-reasoning`), `BIRD_ONLY` restricts the source to [xu3kev/BIRD-SQL-data-train](https://huggingface.co/datasets/xu3kev/BIRD-SQL-data-train), and `LIMIT` caps the number of examples.

In [16]:
# Config (matches prepare_automodel_dataset.py)
VALIDATION_FRACTION = 0.05   # fraction for validation
TEST_FRACTION = 0.1          # fraction for test set (writes test.jsonl); 0 to skip
SPLIT_SEED = 186
INCLUDE_REASONING = True     # False = only BIRD (no reasoning), like --no-reasoning
BIRD_ONLY = False            # if True use only xu3kev/BIRD-SQL-data-train
LIMIT = None                 # set int to limit examples (e.g, 100 for debugging)


In [ ]:
# Using the AutoModel venv from above, so 'datasets' and other deps are available.
# Need this to run the scripts below from within the venv environment. 

if not sys.executable.startswith(venv_dir):
    import glob
    lib_dir = os.path.join(venv_dir, "lib")
    if os.path.isdir(lib_dir):
        for p in sorted(glob.glob(os.path.join(lib_dir, "python*", "site-packages"))):
            sys.path.insert(0, p)
            break

import json
import random
from pathlib import Path
from datasets import load_dataset

BIRD_PROMPT_TEMPLATE = """{schema}

{question}
{evidence}"""

def load_bird_no_reasoning():
    return load_dataset("xu3kev/BIRD-SQL-data-train", split="train")

def load_bird_with_reasoning():
    return load_dataset("meowterspace45/bird-sql-train-with-reasoning", split="train")

def row_to_automodel_example(row, include_reasoning: bool):
    schema = row.get("schema", "")
    question = row.get("question", "")
    evidence = row.get("evidence", "")
    sql = row.get("SQL", "")
    prompt = BIRD_PROMPT_TEMPLATE.format(schema=schema, question=question, evidence=evidence).strip()
    if include_reasoning and "reasoning_trace" in row:
        reasoning = (row["reasoning_trace"] or "").strip()
        answer = f"{reasoning}\n</think>\n\n{sql}".strip() if reasoning else sql.strip()
    else:
        answer = sql.strip()
    return {"input": prompt, "output": answer}

examples = []
print("Loading BIRD-SQL (no reasoning)...")
bird = load_bird_no_reasoning()
for i, row in enumerate(bird):
    if LIMIT and len(examples) >= LIMIT:
        break
    examples.append(row_to_automodel_example(row, include_reasoning=False))
print(f"  Added {len(examples)} examples from BIRD.")

if INCLUDE_REASONING and not BIRD_ONLY:
    print("Loading BIRD with reasoning...")
    start = len(examples)
    try:
        bird_r = load_bird_with_reasoning()
        for i, row in enumerate(bird_r):
            if LIMIT and len(examples) >= LIMIT:
                break
            examples.append(row_to_automodel_example(row, include_reasoning=True))
        print(f"  Added {len(examples) - start} examples from BIRD-with-reasoning.")
    except Exception as e:
        print(f"  Skipping reasoning dataset: {e}")

if LIMIT:
    examples = examples[:LIMIT]

rng = random.Random(SPLIT_SEED)
indices = list(range(len(examples)))
rng.shuffle(indices)
n_test = max(0, int(len(examples) * TEST_FRACTION))
n_val = max(0, int(len(examples) * VALIDATION_FRACTION))
n_train = len(examples) - n_val - n_test
if n_train < 0:
    n_train = 0
    n_val = min(n_val, len(examples) - n_test)
train_examples = [examples[i] for i in indices[:n_train]]
val_examples = [examples[i] for i in indices[n_train : n_train + n_val]]
test_examples = [examples[i] for i in indices[n_train + n_val :]] if n_test > 0 else []

out_dir = Path(DATASET_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
train_path = out_dir / "training.jsonl"
val_path = out_dir / "validation.jsonl"

print(f"Writing {len(train_examples)} training examples to {train_path}...")
with open(train_path, "w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Writing {len(val_examples)} validation examples to {val_path}...")
with open(val_path, "w", encoding="utf-8") as f:
    for ex in val_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
if test_examples:
    test_path = out_dir / "test.jsonl"
    print(f"Writing {len(test_examples)} test examples to {test_path}...")
    with open(test_path, "w", encoding="utf-8") as f:
        for ex in test_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Done. Training: {train_path}  Validation: {val_path}  Test: {test_path}")
else:
    print(f"Done. Training: {train_path}  Validation: {val_path}")


## 3. Finetuning

We run the recipe script from the AutoModel repo root:

```bash
torchrun --nproc-per-node=N examples/llm_finetune/finetune.py --config examples/llm_finetune/nemotron/base-peft-config-cookbook.yaml
```

Make sure you copy the config file (exists at the same place as this notebook, on Nemotron Github) to **`examples/llm_finetune/nemotron`** and the dataset target file to **`nemo_automodel/components/datasets/llm`** within the cloned Automodel directory before running the following cells.

In [ ]:
!pip install matplotlib

In [19]:
CONFIG_NAME = "base-peft-config-cookbook.yaml"
CONFIG_REL = os.path.join("examples", "llm_finetune", "nemotron", CONFIG_NAME)
REPO_PEFT_CONFIG = os.path.join(AUTOMODEL_DIR, CONFIG_REL)

if not os.path.exists(REPO_PEFT_CONFIG):
    raise FileNotFoundError(f"Repo PEFT config not found: {REPO_PEFT_CONFIG}")

PRETRAINED_MODEL_PATH = "..."   # or os.path.join(AUTOMODEL_DIR, ...)
TRAIN_JSONL = os.path.join(DATASET_DIR, "training.jsonl")
VAL_JSONL = os.path.join(DATASET_DIR, "validation.jsonl")

with open(REPO_PEFT_CONFIG, "r", encoding="utf-8") as f:
    content = f.read()
content = content.replace("__PRETRAINED_MODEL_PATH__", PRETRAINED_MODEL_PATH)
content = content.replace("__TRAIN_JSONL__", TRAIN_JSONL)
content = content.replace("__VAL_JSONL__", VAL_JSONL)
# ... any other replacements (checkpoint dir, etc.) ...
with open(REPO_PEFT_CONFIG, "w", encoding="utf-8") as f:
    f.write(content)

In [17]:
# Check all requested GPUs are free and available before running finetuning. Run this cell before the next.
N_DEVICES = 8  # Must match the N_DEVICES in the finetuning cell below; change if using fewer GPUs.
import subprocess
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    lines = [l.strip() for l in out.stdout.strip().split("\n") if l.strip()]
    n_gpus = len(lines)
    print("GPU status (index, memory used, memory total):")
    for i, line in enumerate(lines):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            idx, used_mib, total_mib = parts[0], parts[1].strip() or "0", parts[2].strip() or "0"
            print(f"  GPU {idx}: {used_mib} MiB used / {total_mib} MiB total")
    if n_gpus < N_DEVICES:
        raise RuntimeError(f"Only {n_gpus} GPU(s) visible; finetuning requires {N_DEVICES}. Set N_DEVICES={n_gpus} or free more GPUs.")
    # Consider GPUs "free" if memory used is < 500 MiB (idle). Optionally relax or tighten.
    for i, line in enumerate(lines[:N_DEVICES]):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            used_mib = int(parts[1].strip() or 0)
            if used_mib > 500:
                print(f"Warning: GPU {i} has {used_mib} MiB used; ensure it is free before finetuning.")
    print(f"GPUs: {n_gpus} visible, using {N_DEVICES} for finetuning.")
except FileNotFoundError:
    raise RuntimeError("nvidia-smi not found; cannot check GPUs. Install NVIDIA drivers or run on a GPU node.")

GPU status (index, memory used, memory total):
  GPU 0: 0 MiB used / 81559 MiB total
  GPU 1: 0 MiB used / 81559 MiB total
  GPU 2: 0 MiB used / 81559 MiB total
  GPU 3: 0 MiB used / 81559 MiB total
  GPU 4: 0 MiB used / 81559 MiB total
  GPU 5: 0 MiB used / 81559 MiB total
  GPU 6: 0 MiB used / 81559 MiB total
  GPU 7: 0 MiB used / 81559 MiB total
GPUs: 8 visible, using 8 for finetuning.


In [18]:
FINETUNE_SCRIPT = os.path.join(AUTOMODEL_DIR, "examples", "llm_finetune", "finetune.py")

if not os.path.exists(FINETUNE_SCRIPT):
    raise FileNotFoundError(f"Finetune script not found: {FINETUNE_SCRIPT}")

cmd = [
    "torchrun",
    f"--nproc-per-node={N_DEVICES}",
    FINETUNE_SCRIPT,
    "--config", CONFIG_PATH,
]
# Optional overrides (uncomment and set):
# cmd += ["model.pretrained_model_name_or_path=/path/to/local/model"]
# cmd += [f"dataset.path_or_dataset={os.path.join(DATASET_DIR, 'training.jsonl')}"]

print("Run finetuning with:")
print(" ".join(cmd))

Run finetuning with:
torchrun --nproc-per-node=8 /ephemeral/shadeform/Automodel/examples/llm_finetune/finetune.py --config /ephemeral/shadeform/Automodel/examples/llm_finetune/nemotron/base-peft-config-cookbook.yaml


In [ ]:
# Run finetuning (requires the venv with nemo_automodel installed and CUDA/GPUs).
# Best run in a terminal from the AutoModel repo root (working dir):
#   cd /ephemeral/shadeform/Automodel && source .venv/bin/activate && torchrun --nproc-per-node=8 examples/llm_finetune/finetune.py --config examples/llm_finetune/nemotron/base-peft-config-cookbook.yaml

import subprocess
import re
import matplotlib.pyplot as plt
from datetime import datetime

env = os.environ.copy()
env["DATASET_DIR"] = DATASET_DIR  # some recipes read this for dataset path
# Use venv inside Automodel so torchrun and nemo_automodel are from the right env
venv_bin = os.path.join(AUTOMODEL_DIR, ".venv", "bin")
env["PATH"] = venv_bin + os.pathsep + env.get("PATH", "")

# Stream logs to a file so you can inspect them if something fails
log_dir = os.path.join(AUTOMODEL_DIR, "finetune_logs")
os.makedirs(log_dir, exist_ok=True)
log_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
finetune_log_path = os.path.join(log_dir, f"finetune_{log_stamp}.log")

proc = subprocess.Popen(
    cmd,
    cwd=AUTOMODEL_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
with open(finetune_log_path, "w", encoding="utf-8") as log_f:
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
        log_f.flush()
proc.wait()
print(f"\nLogs saved to: {finetune_log_path}")

if proc.returncode == 0:
    # --- Visualization Logic ---
    steps, losses = [], []
    # Regex matching your specific log line format
    pattern = re.compile(r"step (\d+) .* loss ([\d.]+)")

    with open(finetune_log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                steps.append(int(match.group(1)))
                losses.append(float(match.group(2)))

    if steps:
        plt.figure(figsize=(10, 6))
        plt.plot(steps, losses, label='Training Loss', color='#1f77b4')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('Fine-tuning Convergence')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        
        # Save plot next to the log file
        plot_path = finetune_log_path.replace(".log", ".png")
        plt.savefig(plot_path)
        print(f"Convergence visualization saved to: {plot_path}")
    # ---------------------------

else:
    raise RuntimeError(f"Finetuning exited with code {proc.returncode}. Check logs at {finetune_log_path}")

W0305 03:08:28.773000 2029038 torch/distributed/run.py:803] 
W0305 03:08:28.773000 2029038 torch/distributed/run.py:803] *****************************************
W0305 03:08:28.773000 2029038 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0305 03:08:28.773000 2029038 torch/distributed/run.py:803] *****************************************
/ephemeral/shadeform/Automodel/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type

---
## 4. Deploy: load base + PEFT and run inference

After finetuning, load the base model and PEFT adapter (from the checkpoint resolved by the cell above) and run inference with `transformers` + `peft`. No merge step required — the adapter is applied at load time.

In [47]:
# Follow the "latest" symlink written by train.py to find the most recent run
import os

TRAIN_OUTPUT_PATH = "/home/shadeform/checkpoints_4096_1"
latest_run = os.path.join(train_output_dir, "LATEST")
assert os.path.islink(latest_run), f"No 'latest' symlink found in '{TRAIN_OUTPUT_PATH}'. Run the training step first."

CHECKPOINTS_DIR = os.path.join(os.path.realpath(latest_run), "model")
assert os.path.isdir(CHECKPOINTS_DIR), f"Checkpoints dir not found: '{CHECKPOINTS_DIR}'"

print(f"LoRA checkpoint: {CHECKPOINTS_DIR}")

os.environ["LORA_CHECKPOINT"] = CHECKPOINTS_DIR

LoRA checkpoint: /home/shadeform/checkpoints_4096_1/epoch_0_step_1702/model


In [53]:
# Deploy: run inference in subprocess using Automodel .venv (torch/transformers/peft).
import subprocess
import os

venv_python = os.path.join(AUTOMODEL_DIR, ".venv", "bin", "python")
TEST_DATASET = os.path.join(DATASET_DIR, "test.jsonl")

script = r'''
import json, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = os.environ["BASE_MODEL"]
adapter = os.environ["ADAPTER_PATH"]
test = os.environ["TEST_JSONL"]
tokenizer = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(base, trust_remote_code=True, device_map="auto", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(model, adapter)
with open(test) as f:
    input_text = json.loads(f.readline())["input"]
print("Input:", input_text[:500] + ("..." if len(input_text) > 500 else ""))
out = model.generate(**tokenizer(input_text, return_tensors="pt").to(model.device), max_new_tokens=256, do_sample=False)
print("Output:", tokenizer.decode(out[0], skip_special_tokens=True))
'''

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation (helps 120B OOM)
env["BASE_MODEL"] = PRETRAINED_MODEL_PATH
env["ADAPTER_PATH"] = CHECKPOINTS_DIR
env["TEST_JSONL"] = TEST_DATASET
proc = subprocess.run([venv_python, "-c", script], env=env, cwd=AUTOMODEL_DIR)
if proc.returncode != 0:
    raise RuntimeError(f"Inference exited with code {proc.returncode}")

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 50/50 [00:49<00:00,  1.02it/s]
NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


Input: CREATE TABLE IF NOT EXISTS "Country"
(
    CountryCode                                TEXT not null
        primary key,
    ShortName                                  TEXT,
    TableName                                  TEXT,
    LongName                                   TEXT,
    Alpha2Code                                 TEXT,
    CurrencyUnit                               TEXT,
    SpecialNotes                               TEXT,
    Region                                     TEXT,
    Inco...
Output: CREATE TABLE IF NOT EXISTS "Country"
(
    CountryCode                                TEXT not null
        primary key,
    ShortName                                  TEXT,
    TableName                                  TEXT,
    LongName                                   TEXT,
    Alpha2Code                                 TEXT,
    CurrencyUnit                               TEXT,
    SpecialNotes                               TEXT,
    Region                                

[W305 10:41:48.579880329 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())


---
## Summary

- **Environment:** Use the existing repo and env at `AUTOMODEL_DIR` (no clone or new venv).
- **Data:** Create `training.jsonl`, `validation.jsonl`, and `test.jsonl` in `DATASET_DIR` with `input`/`output` (or your recipe’s expected format).
- **Finetuning:** From the AutoModel repo root, run `torchrun --nproc-per-node=N examples/llm_finetune/finetune.py --config examples/llm_finetune/nemotron/base-peft-config-cookbook.yaml`, overriding `dataset.path_or_dataset` and `model.pretrained_model_name_or_path` as needed.

For **Nemotron-3-Super**, use a config with the correct LoRA targets and a smaller `local_batch_size` / `seq_length` to avoid OOM; enable `activation_checkpointing` if supported.